# gpuless image model builder

Downloads Stable Diffusion XL base 1.0 (about 7 GB) straight into this notebook's output, so it never passes through your own connection.

Settings on the right:

- **Accelerator:** None
- **Internet:** On

Then **Save Version > Save & Run All (Commit)**. When it has finished, create a dataset from the output and call it `gpuless-sdxl`.

The file name has to stay `sd_xl_base_1.0.safetensors`: that is the name the built-in image workflow loads. The model is published by Stability AI under the CreativeML Open RAIL++-M license.

In [ ]:
import os, time, urllib.request

URL = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"
OUT = "/kaggle/working/sd_xl_base_1.0.safetensors"

try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise SystemExit(f"No internet access ({e}). Open the settings on the right and switch Internet on. "
                     "If the switch is locked, verify your phone number under kaggle.com > Settings.")

# Downloaded in Python rather than with wget, so a failure shows up in the cell instead of only in the log.
start, done, step = time.time(), 0, 500 * 2**20
with urllib.request.urlopen(URL, timeout=60) as src, open(OUT, "wb") as dst:
    total = int(src.headers.get("Content-Length", 0))
    next_report = step
    while chunk := src.read(8 * 2**20):
        dst.write(chunk)
        done += len(chunk)
        if done >= next_report:
            print(f"{done / 1e9:5.2f} of {total / 1e9:.2f} GB  ({done / 1e6 / (time.time() - start):.0f} MB/s)", flush=True)
            next_report += step

size = os.path.getsize(OUT)
print(f"done: {size / 1e9:.2f} GB")
if size < 6e9:
    raise SystemExit("the download is too small - Hugging Face may have refused it, see the output above")